# Phase 2 — Test Synthesis Agent

Test the Synthesis Agent on a known set of papers fetched from PubMed.

Verifies:
1. Claude returns well-formed JSON matching the extraction schema
2. Pydantic validation works (rejects bad output, accepts good)
3. Prompt caching is active (check `cache_read_input_tokens` on 2nd+ paper)
4. Extracted fields are sensible

In [ ]:
import sys, logging
sys.path.insert(0, "..")
logging.basicConfig(level=logging.INFO)

from lit_review_agent.tools import search_pubmed
from lit_review_agent.synthesis import synthesize_paper, synthesize_papers
print("Imports OK")

## 1. Fetch test papers from PubMed

Grab 5 wearable-related papers to use as synthesis input.

In [ ]:
query = '(wearable) AND (benchmark OR validation) AND (deep learning OR machine learning)'
papers = search_pubmed(query, max_results=5)

print(f"Fetched {len(papers)} papers")
for i, p in enumerate(papers):
    print(f"  {i+1}. [{p.year}] {p.title[:80]}")

## 2. Synthesize a single paper

Test one paper first. Check JSON output, schema conformance, and cache stats.

In [ ]:
if papers:
    result = synthesize_paper(papers[0])
    print(f"Title: {result.title[:80]}")
    print(f"Task: {result.task}")
    print(f"Devices: {result.devices}")
    print(f"Sensor modalities: {result.sensor_modalities}")
    print(f"Cohort size: {result.cohort_size}")
    print(f"Reference standard: {result.reference_standard}")
    print(f"Split strategy: {result.split_strategy}")
    print(f"Evaluation metrics: {result.evaluation_metrics}")
    print(f"Key findings: {result.key_findings}")
    print(f"Limitations: {result.limitations}")
    print(f"Quality notes: {result.quality_notes}")

## 3. Synthesize all papers

Run on all 5 papers. Watch the logs for `cache_read_input_tokens` — should be >0 starting from the 2nd paper (system prompt cached).

In [ ]:
synthesized = synthesize_papers(papers)

print(f"\nSynthesized {len(synthesized)} papers")
for p in synthesized:
    filled = sum(1 for f in ['task','devices','sensor_modalities','cohort_size',
                              'reference_standard','split_strategy','evaluation_metrics',
                              'key_findings','limitations','quality_notes']
                 if getattr(p, f) not in (None, []))
    print(f"  [{p.year}] {p.title[:60]}... ({filled}/10 fields filled)")

## 4. Inspect full extraction (JSON)

Dump one synthesized paper as JSON to see all fields.

In [ ]:
if synthesized:
    # Show full JSON for first synthesized paper
    print(synthesized[0].model_dump_json(indent=2))